In [14]:
import os
import re
from dotenv import load_dotenv
from pathlib import Path
import pandas as pd
from typing import Optional, Tuple

In [15]:
load_dotenv()

True

In [16]:
BASE_DATASET_PATH = Path(os.getenv("PROJECT_ROOT_DIR")) / "dataset" / os.getenv("BASE_DATASET_NAME")
BASE_DIR_PYTHON_FILES = Path(os.getenv("BASE_DIR_PYTHON_FILES"))
INDENTED_RE = re.compile(r"^indented_(\d+)(?:\.[^.]+)?$") 

In [17]:
print(f"Base dataset path: {BASE_DATASET_PATH}")
print(f"Base directory for Python files: {BASE_DIR_PYTHON_FILES}")

Base dataset path: /home/diogenes/pylingual_colaboration/pylingual_download/code/dataset/pylingual_dataset_summary.csv
Base directory for Python files: /home/diogenes/pylingual_colaboration/pylingual_download/decompiler_workspace


In [18]:

def highest_indented_file(
    directory: Path | str,
    pattern: str = "indented_*"  # change to "indented_*.py" to require .py
) -> Optional[Tuple[Path, int]]:
    """
    Return (path, idx) for the highest-numbered file named like 'indented_{idx}[.ext]'
    in the given directory. Returns None if none found.
    """
    directory = Path(directory)
    best: Optional[Tuple[Path, int]] = None

    for p in directory.glob(pattern):
        if not p.is_file():
            continue
        m = INDENTED_RE.match(p.name)
        if not m:
            continue
        n = int(m.group(1))
        if best is None or n > best[1]:
            best = (p, n)
    return best

In [37]:
pylingual_dataset = pd.read_csv(BASE_DATASET_PATH)

In [26]:
def extract_bytecode_major_minor(src: str) -> Optional[str]:
    m = re.search(r"Bytecode version:\s*(\d+)\.(\d+)", src)
    return f"{m.group(1)}.{m.group(2)}" if m else None

In [38]:
pylingual_dataset['bytecode_version'] = None

for row in pylingual_dataset.itertuples(index=True):
    file_path_base = BASE_DIR_PYTHON_FILES / row.file_hash / "decompiler_output"

    files = highest_indented_file(file_path_base, "indented_*.py")
    if not files:
        continue

    file_path = files[0]

    if file_path.exists():
        with open(file_path, "r") as f:
            code_content = f.read()
            bytecode_version = extract_bytecode_major_minor(code_content)

        pylingual_dataset.loc[row.Index, "bytecode_version"] = bytecode_version

In [39]:
pylingual_dataset.groupby("bytecode_version").size()

bytecode_version
3.10     27162
3.11     42682
3.12    155719
3.13     15969
3.6       2441
3.7      18989
3.8      14061
3.9      16252
dtype: int64

In [40]:
pylingual_dataset.to_csv(BASE_DATASET_PATH, index=False)